# Call CNVs, then see them in context

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmdcolin/jbrowse-anywidget/blob/main/examples/05_cnv_calling.ipynb)

The notebook loop: **run an analysis, drop the result onto the genome.** Here we segment binned read-depth into copy-number gains and losses, then load the calls as a track — no file written. The synthetic depth carries a focal amplification at *ERBB2* (HER2), the classic breast-cancer CNV.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/cmdcolin/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## The signal: binned log2 depth ratio

Stand-in for a tumor/normal coverage ratio over a stretch of chr17. Swap it for your own binned depth (`cnvkit`, `mosdepth`, …) as long as it has chrom/start/end and a log2 column.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
chrom, start, end, binsize = "17", 39_000_000, 40_200_000, 10_000
bin_starts = np.arange(start, end, binsize)
log2 = rng.normal(0, 0.25, size=bin_starts.size)

# a focal ERBB2/HER2 amplification, plus a nearby deletion
log2[(bin_starts >= 39_680_000) & (bin_starts < 39_740_000)] += 2.4
log2[(bin_starts >= 39_180_000) & (bin_starts < 39_260_000)] -= 1.4

bins = pd.DataFrame(
    {
        "chrom": chrom,
        "start": bin_starts,
        "end": bin_starts + binsize,
        "log2": log2.round(3),
    }
)
bins.head()

## Call segments

Smooth, threshold into gain/loss/neutral, and merge adjacent like-state bins into segments — a toy segmenter standing in for CBS/HMM.

In [ ]:
GAIN, LOSS = 0.4, -0.4
smooth = bins["log2"].rolling(5, center=True, min_periods=1).median().to_numpy()
state = np.where(smooth > GAIN, "gain", np.where(smooth < LOSS, "loss", "neutral"))

segments, i = [], 0
while i < len(bins):
    j = i
    while j + 1 < len(bins) and state[j + 1] == state[i]:
        j += 1
    if state[i] != "neutral":
        segments.append(
            {
                "chrom": chrom,
                "start": int(bins["start"][i]),
                "end": int(bins["end"][j]),
                "state": state[i],
                "mean_log2": round(float(smooth[i : j + 1].mean()), 2),
            }
        )
    i = j + 1

calls = pd.DataFrame(segments)
calls

## Load signal + calls onto the genome

Two `add_features` tracks: the per-bin log2 (red gain / blue loss) and the called segments on top. `color` is a jexl expression over each feature's own columns.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, make_assembly

grch38 = make_assembly(
    "GRCh38",
    "https://s3.amazonaws.com/jbrowse.org/genomes/GRCh38/fasta/GRCh38.fa.gz",
    aliases=["hg38"],
)
view = LinearGenomeView(assembly=grch38, location="17:39,000,000..40,200,000")
view.add_features(
    bins,
    name="log2 depth ratio",
    color="jexl:get(feature,'log2') > 0.4 ? '#c62828' : get(feature,'log2') < -0.4 ? '#1565c0' : '#cfcfcf'",
)
view.add_features(
    calls,
    name="CNV calls",
    color="jexl:get(feature,'state') == 'gain' ? '#c62828' : '#1565c0'",
)
view

Zoom to *ERBB2* to land on the amplified segment (drives the view from Python):

In [ ]:
view.location = "17:39,650,000..39,770,000"